In [0]:
-- 03_gold_01_core
-- Layer: gold (foundational truth)
-- Purpose: answer question such as "what is the current state of our transactions and entities?"
-- Method: joining silver entities into flattened, high-performance star schema tables.

CREATE SCHEMA IF NOT EXISTS olist_maas_pipeline.gold;

In [0]:
-- Intermediate processing for internal views
-- Aggregate payments to Order level
CREATE OR REPLACE VIEW olist_maas_pipeline.gold.int_payments_summary AS
SELECT 
    order_id
    , SUM(payment_value) AS total_payment
    , MAX(payment_installments) AS max_installments
    , COLLECT_LIST(NAMED_STRUCT(
        'type', payment_type
        , 'value', payment_value
        , 'installments', payment_installments
      )) AS payment_details_json
    , COUNT(*) AS payment_count
FROM olist_maas_pipeline.silver.sales_order_payments
GROUP BY order_id;

-- Deduplicate reviews (Keep the most recent feedback per order)
CREATE OR REPLACE VIEW olist_maas_pipeline.gold.int_reviews_latest AS
SELECT review_id, order_id, review_score, review_comment_message, review_creation_date
FROM (
    SELECT *
        , ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_creation_date DESC) AS rn
    FROM olist_maas_pipeline.silver.sales_order_reviews
    WHERE order_id IS NOT NULL
)
WHERE rn = 1

In [0]:
-- Core fact tables
-- Central transaction table
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.fact_orders
USING DELTA
AS
SELECT 
    o.order_id
    , o.customer_id
    , o.customer_unique_id
    , o.customer_state
    , o.region_name
    , o.logistics_tier
    , o.breaking_point_days
    , o.order_status
    -- Status Indicators
    , CASE WHEN o.order_status = 'delivered' THEN TRUE ELSE FALSE END AS is_completed
    , CASE WHEN o.order_status = 'canceled' THEN TRUE ELSE FALSE END AS is_cancelled
    -- Dates
    , DATE(o.order_purchase_timestamp) AS purchase_date
    , DATE(o.order_delivered_customer_date) AS delivery_date
    , DATE(o.order_estimated_delivery_date) AS estimated_date
    -- Performance Metrics
    , o.actual_delivery_days
    , DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) AS sla_variance_days
    -- Performance Flags
    , CASE 
        WHEN o.order_status != 'delivered' THEN NULL
        WHEN DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) <= 0 THEN TRUE 
        ELSE FALSE 
      END AS is_on_time
    , CASE 
        WHEN o.order_status != 'delivered' THEN NULL
        WHEN DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) >= o.breaking_point_days THEN TRUE 
        ELSE FALSE 
      END AS is_at_breaking_point
    -- Enrichment Data (JSON & Reviews)
    , r.review_score
    , p.total_payment
    , p.payment_details_json
    , p.payment_count
    -- Time Dimensions
    , DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS year_month
    , YEAR(o.order_purchase_timestamp) AS order_year
FROM olist_maas_pipeline.silver.sales_orders o
LEFT JOIN olist_maas_pipeline.gold.int_payments_summary p ON o.order_id = p.order_id
LEFT JOIN olist_maas_pipeline.gold.int_reviews_latest r ON o.order_id = r.order_id;

-- fact order items (multi-seller orders)
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.fact_order_items
USING DELTA
AS
SELECT 
    i.order_id
    , i.order_item_id
    , i.product_id
    , i.seller_id
    , COALESCE(i.product_category_name, 'unknown') AS product_category_name
    , i.price
    , i.freight_value
    , i.product_weight_g
    -- Revenue Calculation
    , (i.price + i.freight_value) AS gmv
    , ROUND((i.price + i.freight_value) * 0.15, 2) AS platform_revenue
    -- Logistics Complexity
    , CASE 
        WHEN i.product_weight_g >= 10000 THEN 'HEAVY'
        WHEN i.product_weight_g >= 2000 THEN 'MEDIUM'
        ELSE 'LIGHT'
      END AS weight_category
FROM olist_maas_pipeline.silver.sales_order_items i;

In [0]:
-- Core Dimension Tables
-- dim date
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT MIN(purchase_date) AS min_date, MAX(purchase_date) AS max_date
    FROM olist_maas_pipeline.gold.fact_orders
),
date_spine AS (
    SELECT explode(sequence(min_date, max_date, interval 1 day)) AS date FROM date_range
)
SELECT 
    CAST(DATE_FORMAT(date, 'yyyyMMdd') AS INT) AS date_key
    , date
    , YEAR(date) AS year
    , QUARTER(date) AS quarter
    , MONTH(date) AS month
    , DATE_FORMAT(date, 'MMMM') AS month_name
    , DAYOFWEEK(date) AS day_of_week
    , DATE_FORMAT(date, 'EEEE') AS day_name
    , CASE WHEN DAYOFWEEK(date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM date_spine;

-- dim product
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.dim_product
USING DELTA
AS
SELECT 
    product_id
    , FIRST(product_category_name) AS category
    , FIRST(product_weight_g) AS weight_g
    , COUNT(DISTINCT order_id) AS lifetime_orders
    , SUM(price) AS lifetime_sales
FROM olist_maas_pipeline.gold.fact_order_items
GROUP BY product_id;